# EchoROI: Inference Demo

This notebook demonstrates how to use the **echoroi** package for:
1. Loading the pretrained model
2. Predicting ROI masks on echocardiogram images
3. Extracting the ROI crop
4. De-identifying images (masking outside the ROI)

---

## 1. Setup

In [3]:
import os
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Navigate to project root regardless of how the kernel was launched
notebook_dir = Path(os.path.abspath("")).resolve()
if notebook_dir.name == "notebooks":
    os.chdir(notebook_dir.parent)
elif not (notebook_dir / "models").exists():
    # Try one level up
    if (notebook_dir.parent / "models").exists():
        os.chdir(notebook_dir.parent)

print(f"Working directory: {os.getcwd()}")
assert Path("models/echoroi_unified.keras").exists(), "Model not found — check working directory"

from echoroi.inference import UNetPredictor
print("echoroi loaded successfully")

Working directory: /Volumes/G-DRIVE PRO/UNET-Ultrasound-ROI-Segmentation
echoroi loaded successfully


## 2. Load Pretrained Model

The `UNetPredictor` class handles model loading and provides a simple inference API.

In [4]:
predictor = UNetPredictor("models/echoroi_unified.keras")
print(f"Model loaded: {predictor.model.count_params():,} parameters")

Loading model from: models/echoroi_unified.keras


TypeError: Could not locate function '_dice_coefficient'. Make sure custom classes are decorated with `@keras.saving.register_keras_serializable()`. Full object config: {'module': 'builtins', 'class_name': 'function', 'config': '_dice_coefficient', 'registered_name': 'function'}

## 3. Single Image Prediction

Predict the ROI mask for a single echocardiogram image.

In [ ]:
import glob

# Pick a sample image
sample_images = sorted(glob.glob("data/images/*.png"))
sample_path = sample_images[42]  # Arbitrary sample
print(f"Input: {sample_path}")

# Predict
mask, confidence = predictor.predict_single_image(sample_path)
print(f"Mask shape: {mask.shape}")
print(f"Confidence: {confidence:.4f}")

# Visualize
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

import cv2
original = cv2.imread(sample_path, cv2.IMREAD_GRAYSCALE)
axes[0].imshow(original, cmap='gray')
axes[0].set_title('Original Image')
axes[0].axis('off')

axes[1].imshow(mask, cmap='gray')
axes[1].set_title(f'Predicted Mask (conf: {confidence:.3f})')
axes[1].axis('off')

# Overlay
overlay = cv2.cvtColor(original, cv2.COLOR_GRAY2RGB) if len(original.shape) == 2 else original.copy()
mask_resized = cv2.resize(mask.astype(np.float32), (original.shape[1], original.shape[0]))
overlay_mask = (mask_resized > 0.5).astype(np.uint8)
overlay[..., 1] = np.clip(overlay[..., 1].astype(np.int32) + overlay_mask * 60, 0, 255).astype(np.uint8)
axes[2].imshow(overlay)
axes[2].set_title('Overlay')
axes[2].axis('off')

plt.suptitle('Single Image Prediction', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. ROI Extraction

Crop the image to just the detected ROI region.

In [ ]:
roi_crop = predictor.extract_roi(sample_path)
print(f"ROI crop shape: {roi_crop.shape}")

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(original, cmap='gray')
axes[0].set_title('Original')
axes[0].axis('off')

axes[1].imshow(roi_crop, cmap='gray')
axes[1].set_title('Extracted ROI')
axes[1].axis('off')

plt.tight_layout()
plt.show()

## 5. De-identification

Mask everything outside the ROI to remove patient identification information
(text overlays, ECG traces, institutional logos).

In [ ]:
deidentified = predictor.apply_mask_for_deidentification(sample_path)
print(f"De-identified shape: {deidentified.shape}")

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(original, cmap='gray')
axes[0].set_title('Original (may contain PHI)')
axes[0].axis('off')

axes[1].imshow(deidentified, cmap='gray')
axes[1].set_title('De-identified (ROI only)')
axes[1].axis('off')

plt.suptitle('De-identification via ROI Masking', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Batch Processing

Process multiple images and display results.

In [ ]:
batch_images = sample_images[10:16]  # 6 samples

fig, axes = plt.subplots(len(batch_images), 3, figsize=(15, 4 * len(batch_images)))

for i, img_path in enumerate(batch_images):
    original = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
    mask, conf = predictor.predict_single_image(img_path)
    deident = predictor.apply_mask_for_deidentification(img_path)

    axes[i, 0].imshow(original, cmap='gray')
    axes[i, 0].set_title(os.path.basename(img_path)[:25], fontsize=9)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(mask, cmap='gray')
    axes[i, 1].set_title(f'Mask (conf: {conf:.3f})', fontsize=9)
    axes[i, 1].axis('off')

    axes[i, 2].imshow(deident, cmap='gray')
    axes[i, 2].set_title('De-identified', fontsize=9)
    axes[i, 2].axis('off')

plt.suptitle('Batch Inference Results', fontsize=14, fontweight='bold', y=1.005)
plt.tight_layout()
plt.show()

## 7. Inference Speed

Benchmark single-image inference latency.

In [ ]:
latency = predictor.benchmark_inference_speed(n_iterations=100)
print(f"\nAverage inference latency: {latency:.1f} ms per image")

## Summary

The `echoroi` package provides a simple API for cardiac ultrasound ROI segmentation:

```python
from echoroi.inference import UNetPredictor

predictor = UNetPredictor("models/echoroi_unified.keras")

# Predict mask
mask, confidence = predictor.predict_single_image("image.png")

# Extract ROI crop
roi = predictor.extract_roi("image.png")

# De-identify (mask outside ROI)
safe = predictor.apply_mask_for_deidentification("image.png")
```

For ONNX deployment, see notebook 02. For CLI usage: `echoroi predict --help`